In [ ]:
%pip install pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 1.9 MB/s  0:00:05m 1.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 2.6 MB/s  0:00:022.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]━━━━ 3/4 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:

import pandas as pd
import re

MANIFEST_PATH = '/Users/shaswatshrey/Development/MajorProject/manifest.csv'
PROCESS_TABLE_PATH = '/Users/shaswatshrey/Development/MajorProject/process_table.csv'

In [5]:
def parse_layer_file(path):
    df = pd.read_csv(path)
    df.columns = ["hit_id","date","time","PARA1","CH","RISE","COUNTS","ENERGY","DURATION",
                  "AMP","A_FRQ","RMS","ASL","PCNTS","THR","R_FRQ","I_FRQ","SIG_STRNGTH",
                  "ABS_ENERGY","FRQ_C","P_FRQ"]
    numeric_cols = df.columns[5:]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df

In [6]:
import pandas as pd
manifest = pd.read_csv('/Users/shaswatshrey/Development/MajorProject/manifest.csv')
print(manifest.columns.tolist())
print(manifest.shape)
print(manifest.head())

print([repr(c) for c in manifest.columns])

['filepath', 'sample_id', 'layers_in_file']
(144, 3)
                                           filepath  sample_id layers_in_file
0  SAMPLE 1 TO 8/SAMPLE 1/files/Layer_11_and_12.csv          1          11,12
1  SAMPLE 1 TO 8/SAMPLE 1/files/Layer_13_and_14.csv          1          13,14
2  SAMPLE 1 TO 8/SAMPLE 1/files/Layer_15_and_16.csv          1          15,16
3  SAMPLE 1 TO 8/SAMPLE 1/files/Layer_17_and_18.csv          1          17,18
4  SAMPLE 1 TO 8/SAMPLE 1/files/Layer_19_and_20.csv          1          19,20
["'filepath'", "'sample_id'", "'layers_in_file'"]


In [7]:

manifest = pd.read_csv(MANIFEST_PATH, sep=',')
process_table = pd.read_csv(PROCESS_TABLE_PATH, sep=',')
manifest.columns = manifest.columns.str.strip()
process_table.columns = process_table.columns.str.strip()

all_hits = []
for _, row in manifest.iterrows():
    df = parse_layer_file(row.filepath)
    layer_ids = [int(x) for x in str(row.layers_in_file).split(",")]

    if len(layer_ids) == 1:
        df["layer_id"] = layer_ids[0]
        segments = [df]
    else:
        split_point = len(df) // len(layer_ids)
        segments = []
        for i, lid in enumerate(layer_ids):
            seg = df.iloc[i*split_point:(i+1)*split_point].copy()
            seg["layer_id"] = lid
            segments.append(seg)

    for seg in segments:
        seg["sample_id"] = row.sample_id
        all_hits.append(seg)

hit_df = pd.concat(all_hits, ignore_index=True)
hit_df = hit_df.merge(process_table, on="sample_id", how="left")

print(hit_df.shape)
print(hit_df.groupby("sample_id")["layer_id"].nunique())   # MUST show 37 for every sample_id
print(hit_df.isna().sum().sum())
hit_df.to_csv('hit_df.csv', index=True)

(999192, 26)
sample_id
1    38
2    40
3    30
4    32
5    34
6    32
7    42
8    40
Name: layer_id, dtype: int64
0


In [8]:
layer_table = hit_df.groupby(
    ["sample_id", "layer_id", "level", "travel_speed", "heat_input"]
).agg(
    ABS_ENERGY_mean = ("ABS_ENERGY", "mean"),
    ABS_ENERGY_std  = ("ABS_ENERGY", "std"),
    RISE_mean       = ("RISE", "mean"),
    n_hits          = ("ABS_ENERGY", "count")
).reset_index()

print(layer_table.shape)                          # expect (333, 9) -> 9 samples x 37 layers
print(layer_table.groupby("sample_id").size())     # MUST show 37 for every sample_id
print(layer_table.isna().sum().sum())              # MUST be 0
print(layer_table.head())
layer_table.to_csv('lay_table.csv', index=False)

(288, 9)
sample_id
1    38
2    40
3    30
4    32
5    34
6    32
7    42
8    40
dtype: int64
0
   sample_id  layer_id  level  travel_speed  heat_input  ABS_ENERGY_mean  \
0          1         1      1            10       260.1     9.009400e+05   
1          1         2      1            10       260.1     8.763473e+05   
2          1         3      1            10       260.1     8.287545e+05   
3          1         4      1            10       260.1     1.183784e+06   
4          1         5      1            10       260.1     1.758617e+06   

   ABS_ENERGY_std     RISE_mean  n_hits  
0    1.204488e+06  22631.024005    1458  
1    6.334267e+05  28331.185185    1458  
2    1.081971e+06  22630.118617    1273  
3    3.096503e+05  26283.597800    1273  
4    3.849057e+05  26214.410185    1080  


In [12]:
block_table = layer_table.groupby(
    ["sample_id", "level", "travel_speed", "heat_input"]
).agg(
    ABS_ENERGY_blockstd = ("ABS_ENERGY_mean", "std"),   # variability of avg energy ACROSS the 37 layers
    RISE_blockmean       = ("RISE_mean", "mean")          # average onset behavior across the block
).reset_index()

print(block_table.shape)   # expect exactly (9, 6)
print(block_table)
hardness_lookup = {
    1: 310.2,   # <-- fill in your 9 actual measured hardness values here
    2: 304.7,
    3: 291.6,
    4: 305.1,
    5: 313.3,
    6: 307.2,
    7: 303.2,
    8: 305.6,
    9: 293.4,
}

block_table["hardness"] = block_table["sample_id"].map(hardness_lookup)
print(block_table)
print(block_table.isna().sum())
block_table.to_csv('block_table.csv', index=False)

(8, 6)
   sample_id  level  travel_speed   heat_input  ABS_ENERGY_blockstd  \
0          1      1            10        260.1        847439.671305   
1          2      2            10       307.79          5180.954958   
2          3      1             8       325.13        178708.707270   
3          4      2             8       384.73        271019.267074   
4          5      3            10       450.18         59497.714143   
5          6      3             8       216.75           144.815649   
6          7      1            12       256.49        471647.013706   
7          8      2            12       300.12        486947.946584   

   RISE_blockmean  
0    13766.758853  
1     1752.865761  
2    18467.305567  
3    21863.196477  
4     6863.514763  
5      473.493832  
6    24525.343173  
7    23300.696556  
   sample_id  level  travel_speed   heat_input  ABS_ENERGY_blockstd  \
0          1      1            10        260.1        847439.671305   
1          2      2            

In [13]:
print(layer_table.shape)
print(layer_table.groupby("sample_id").size())
print(layer_table.isna().sum().sum())
print(block_table)

(288, 9)
sample_id
1    38
2    40
3    30
4    32
5    34
6    32
7    42
8    40
dtype: int64
0
   sample_id  level  travel_speed   heat_input  ABS_ENERGY_blockstd  \
0          1      1            10        260.1        847439.671305   
1          2      2            10       307.79          5180.954958   
2          3      1             8       325.13        178708.707270   
3          4      2             8       384.73        271019.267074   
4          5      3            10       450.18         59497.714143   
5          6      3             8       216.75           144.815649   
6          7      1            12       256.49        471647.013706   
7          8      2            12       300.12        486947.946584   

   RISE_blockmean  hardness  
0    13766.758853     310.2  
1     1752.865761     304.7  
2    18467.305567     291.6  
3    21863.196477     305.1  
4     6863.514763     313.3  
5      473.493832     307.2  
6    24525.343173     303.2  
7    23300.696556     

In [14]:
import sys
import platform

print("Python:", sys.executable)
print("Architecture:", platform.machine())

Python: /Users/shaswatshrey/Development/MajorProject/.venv/bin/python
Architecture: arm64


In [2]:
import numpy
import scipy
import sklearn

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy: 2.4.6
SciPy: 1.17.1
Scikit-learn: 1.9.1


In [18]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ... your existing run_lobo code ...
y = block_table["hardness"]

# Full model: process + AE features
X_full = layer_table[["heat_input", "ABS_ENERGY_mean", "RISE_mean"]]

# Baseline: process parameter alone
X_baseline = block_table[["heat_input"]]

def run_lobo(X, y, label):
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50), cv=LeaveOneOut()))
    scores = cross_val_score(model, X, y, cv=LeaveOneOut(), scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    model.fit(X, y)
    ridge = model.named_steps["ridgecv"]
    print(f"--- {label} ---")
    print("LOBO-CV MAE:", round(mae, 3))
    print("Best alpha:", ridge.alpha_)
    print("Coefficients:", dict(zip(X.columns, ridge.coef_.round(3))))
    print()
    return mae

mae_baseline = run_lobo(X_baseline, y, "Baseline (heat_input only)")
mae_full     = run_lobo(X_full, y, "Full (heat_input + AE features)")

print("Improvement:", round(mae_baseline - mae_full, 3), 
      "(positive = AE features helped)")

--- Baseline (heat_input only) ---
LOBO-CV MAE: 5.99
Best alpha: 0.001
Coefficients: {'heat_input': np.float64(1.053)}



ValueError: Found input variables with inconsistent numbers of samples: [288, 8]